In [1]:
from sympy import *
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [2]:
g=Symp_symb(7)
C=g.cochain_complex
K=IndexedBase('K')
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis
P=RegularCartanGeometry(g,'eta')
eta=IndexedBase('eta')
P.fund_invars=[eta[3,9,6],eta[6,9,9],eta[3,9,4]]
D=Distr_of_constant_symbol(g,-P.curvature)
P.curvature=ds_subs(P.curvature,{eta[3,9,6]:{tuple():0},eta[3,9,4]:{tuple():0}},D)
D=Distr_of_constant_symbol(g,-P.curvature)

In [3]:
def ds_add_key(key,val,ds_dict,Distr):
    """adds the key, value pair to the differential substitution dictionary ds_dict,
    back substituting if necessary. Overwrites existing key if already present"""
    b_key=key.base[key.indices[0:3]]
    post=key.indices[3:len(key.indices)]

    # first back substitute
    ds_back_substitute(ds_dict,{b_key:{post:val}},Distr,-oo)
    
    # then add the key if it's not redundant
    new_expr=ds_subs(key-val,ds_dict,Distr)
    if new_expr!=key-val:
        add_expr_to_ds_dict(new_expr,ds_dict,Distr)
    else:
        if not b_key in ds_dict: ds_dict[b_key]={}
        ds_dict[b_key][post]=val

def ds_back_substitute(d1,d2,Distr,min_wght):
    """Back substitutes d2 into d1
    INPUTS:
    * 'd1,d2' - differential substitution dictionaries
    * 'Distr' - a distribution of constant symbol
    * 'min_wght' - the minimal weight of the keys of d2"""

    # First, remove proper children of the new keys
    removed_ids=[]
    keys_to_remove=[]
    for k2 in d2:
        if k2 in d1:
            for i2 in d2[k2]:
                for i1 in d1[k2]:
                    if i1[0:len(i2)]==i2 and len(i2)<len(i1):
                        removed_ids.append(k2.base[*k2.indices,*i1]-d1[k2][i1])
                        keys_to_remove.append((k2,i1))
    for pair in keys_to_remove: del d1[pair[0]][pair[1]]

    for a in d1:
        for b in d1[a]:
            my_wght=wght_of_ind(a,Distr.Tanaka_symbol)
            for i in b:
                my_wght+=-Distr.Tanaka_symbol.basis[i].wght
            if my_wght>=min_wght:
                d1[a][b]=ds_subs(d1[a][b],d2,Distr)
    
    # Add back the removed identities
    for rI in removed_ids:
        rI=ds_subs(ds_subs(rI,d2,Distr),d1,Distr)
        add_expr_to_ds_dict(rI,d1,Distr)

def add_expr_to_ds_dict(expr,ds_dict,Distr):
    """"Computes a substitution from the equation expr==0, substituting back into ds_dict.
    Assumes expr has already been substituted with ds_dict.
    Used, for example, to reprocess a key like K[3,4,5,3] when K[3,4,5] is processed."""
    curr_JI=expr
    # curr_JI may already be in the ideal ds_dict
    if curr_JI==0: return None

    # There may be no linear monomials to isolate without branching
    curr_K=find_a_linear_term(curr_JI)
    if curr_K==None:
        curr_JI=factor(curr_JI).as_coeff_Mul()[1].as_base_exp()[0]
        curr_K=find_a_linear_term(curr_JI)
        if curr_K==None:
            # For the moment, just pick a random indexed object with longish index list
            L=list(Indexed_obj_in_expr(curr_JI))
            curr_K=L[0]
            for a in L:
                if len(curr_K.indices)<len(a.indices): curr_K=a

    list_to_avoid=[]
    # # (2,6) case
    if len(Distr.basis)==8: list_to_avoid=[(3,9,6),(3,9,4),(3,9,8),(3,9,9),(4,8,6),(4,8,7),(4,7,5)]
    # # (2,7) case
    if len(Distr.basis)==10: list_to_avoid=[(3,11,8),(3,11,6),(3,11,4),(3,11,11),(3,11,10)]

    # # Note: This is not airtight; if the only linear terms are from list_to_avoid this will give problems
    # # Really, it should be fine, though
    if curr_K.indices[0:3] in list_to_avoid:
        F=list(find_linear_terms(curr_JI))
        # We'll try to avoid using the Wilcinski invariants whenever possible
        i=0
        j=0
        while i<len(F):
            if F[i].indices[0:3] in list_to_avoid: i+=1
            else:
                j=i
                i=len(F)
        curr_K=F[j]

    
    # Solve; add to ds_dict
    try: s=solve(curr_JI,curr_K)[0]
    except:
        temp={Kijk:Symbol(str(Kijk)) for Kijk in Indexed_obj_in_expr(curr_JI)}
        temp_inv={temp[Kijk]:Kijk for Kijk in temp}
        s=solve(curr_JI.xreplace(temp),Symbol(str(curr_K)))[0].xreplace(temp_inv)
    ds_add_key(curr_K,s,ds_dict,Distr)

In [4]:
def der_term(P,i0,i1,i2):
    """Returns the derivative term for the component of the Bianchi identity
    applied to (i0,i1,i2), which is an element of P.symbol."""
    g=P.symbol
    X0,X1,X2=[g.basis[a] for a in [i0,i1,i2]]
    w0,w1,w2=[None,None,None]
    try: w0=g.ext_alg.elt_from_cd({(str(X1),str(X2)):1})
    except(KeyError): pass
    try: w1=g.ext_alg.elt_from_cd({(str(X2),str(X0)):1})
    except(KeyError): pass
    try: w2=g.ext_alg.elt_from_cd({(str(X0),str(X1)):1})
    except(KeyError): pass

    r=g.elt()
    if w0!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w0),i0)
    if w1!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w1),i1)
    if w2!=None: r+=P.fund_der(P.curvature.apply_cochain_map(w2),i2)
    return r

def cb_term(P,i0,i1,i2):
    """returns the coboundary of P.curvature applied to i0,i1,i2 elements of P.symbol.basis.
    This is needed because we care about the value as a cochain in C(g,g), not just C(m,g)
    (at least for the purpose of checks)"""

    r=P.symbol.elt()
    X0,X1,X2=[P.symbol.basis[a] for a in [i0,i1,i2]]
    
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        w0=Y0.negative_projection().cast_as_ext_elt().wedge(Y1.negative_projection().cast_as_ext_elt())
        r+=P.curvature.apply_cochain_map(w0).ad(Y2)
        w1=Y0.ad(Y1).negative_projection().cast_as_ext_elt().wedge(Y2.negative_projection().cast_as_ext_elt())
        r+=P.curvature.apply_cochain_map(w1)
    return -r

# What if instead of computing these individually, I just computed Gerstenhaber square
# of curvature once, then 
def gerst_term(P,i0,i1,i2):
    X0,X1,X2=[P.symbol.basis[a] for a in [i0,i1,i2]]
    r=P.symbol.elt()
    for i in range(3):
        Y0,Y1,Y2=[X0,X1,X2,X0,X1][i:i+3]
        Y0=Y0.negative_projection().cast_as_ext_elt()
        Y1=Y1.negative_projection().cast_as_ext_elt()
        Y2=Y2.negative_projection().cast_as_ext_elt()
        w=P.curvature.apply_cochain_map(Y0.wedge(Y1)).negative_projection().cast_as_ext_elt()
        r+=P.curvature.apply_cochain_map(w.wedge(Y2))
    return r

def Bianchi(P,i0,i1,i2,subdivide=False):
    r0=cb_term(P,i0,i1,i2)
    r1=der_term(P,i0,i1,i2)
    r2=gerst_term(P,i0,i1,i2)
    if subdivide: return (r0, r1, r2)
    return r0+r1+r2
    # if subdivide: return (cb_term(P,i0,i1,i2), der_term(P,i0,i1,i2), gerst_term(P,i0,i1,i2))
    # return cb_term(P,i0,i1,i2)+der_term(P,i0,i1,i2)+gerst_term(P,i0,i1,i2)

In [5]:
def subs_needed(expr,ds_dict):
    I=Indexed_obj_in_expr(expr)
    for A in I:
        pre=A.indices[0:3]
        post=A.indices[3:len(A.indices)]
        if eta[pre] in ds_dict:
            for i in range(len(post)+1):
                if post[0:i] in ds_dict[eta[pre]]: return True
    return False

def ds_subs_needed(ds_dict):
    for k in ds_dict:
        for j in ds_dict[k]:
            if subs_needed(ds_dict[k][j],ds_dict): 
                print(k,j)
                return True
    return False

### Computations

In [6]:
tuples_by_wght={}
for i in range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        for k in range(j+1,len(g.basis)):
            for m in range(len(g.basis)):
                w=-g.basis[i].wght-g.basis[j].wght-g.basis[k].wght+g.basis[m].wght
                if w not in tuples_by_wght: tuples_by_wght[w]=[]
                tuples_by_wght[w].append((i,j,k,m))

for w in range(min(list(tuples_by_wght.keys())),max(list(tuples_by_wght.keys()))+1):
    print(w,'-->',len(tuples_by_wght[w]))

-3 --> 1
-2 --> 2
-1 --> 5
0 --> 8
1 --> 13
2 --> 19
3 --> 27
4 --> 34
5 --> 43
6 --> 49
7 --> 55
8 --> 58
9 --> 59
10 --> 55
11 --> 50
12 --> 42
13 --> 34
14 --> 25
15 --> 17
16 --> 10
17 --> 6
18 --> 3
19 --> 1


In [7]:
Bianchi_dict={}
not_added=[]

def compute_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        time0=time.time()
        print('Computing', t)
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else: 
            time1=time.time()
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
            print('    Bianchi computed in',hrs_min_sec(time.time()-time1))
        time2=time.time()
        to_solve=ds_subs(zero_elt.vec[m],Bianchi_dict,D)
        print('    to_solve computed in',hrs_min_sec(time.time()-time2))
        if simplify(to_solve)!=0:
            time3=time.time()
            s=find_a_linear_term(to_solve,P.fund_invars)
            if s==None: s=find_a_linear_term(to_solve)
            if s==None:
                print('no linear term in',t)
                not_added.append(t)
            else:
                sol=solve(to_solve,s,dict=True)[0]
                print('    Solving complete in',hrs_min_sec(time.time()-time3))
                time4=time.time()
                for a in sol: 
                    ds_add_key(a,sol[a],Bianchi_dict,D)
                print('    Substitution complete in',hrs_min_sec(time.time()-time4))
        print('   ',t,'computed in',hrs_min_sec(time.time()-time0))
        # Notice that D.curv = -P.curvature, since I switched sign conventions

def check_Bianchi(w):
    for t in tuples_by_wght[w]:
        i,j,k,m=t
        if (i,j,k) in P.Bianchi_cache:
            zero_elt=P.Bianchi_cache[(i,j,k)]
        else:
            zero_elt=Bianchi(P,i,j,k)
            P.Bianchi_cache[(i,j,k)]=zero_elt
        r=simplify(ds_subs(zero_elt.vec[m],Bianchi_dict,D))
        if r!=0: print(r)

In [8]:
for w in range(1,10):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

--------------------- Weight 1 ---------------------
Computing (3, 4, 5, 6)
    Bianchi computed in 3.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 5, 6) computed in 3.0 sec
Computing (3, 4, 6, 7)
    Bianchi computed in 2.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 6, 7) computed in 2.0 sec
Computing (3, 4, 7, 8)
    Bianchi computed in 4.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 7, 8) computed in 4.0 sec
Computing (3, 4, 8, 9)
    Bianchi computed in 8.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 8, 9) computed in 8.0 sec
Computing (3, 4, 9, 10)
    Bianchi computed in 22.0 sec
    to_solve computed in 0.0 sec
    Solving complete in 0.0 sec
    Substitution complete in 0.0 sec
    (3, 4, 9

In [55]:
# wght_8_Bianchi=copy.deepcopy(Bianchi_dict)
# Bianchi_dict=copy.deepcopy(wght_8_Bianchi)

In [ ]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

In [ ]:
for w in range(10,13):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

In [ ]:
for w in range(13,17):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

In [ ]:
for w in range(17,19):
    time0=time.time()
    print('--------------------- Weight',w,'---------------------')
    compute_Bianchi(w)
    print('Bianchi wght',w,'computed in', hrs_min_sec(time.time()-time0))
    P.update_fund_ders(Bianchi_dict,D)
    P.curvature=ds_subs(P.curvature,Bianchi_dict,D)
    D.curv=-P.curvature
    print('Weight',w,'complete in',hrs_min_sec(time.time()-time0))

In [ ]:
print(ds_subs_needed(Bianchi_dict))
print(not_added)

### Syzygies

In [22]:
syzygies_by_wght={}
for k in P.fund_invars:
    if k in Bianchi_dict:
        for j in Bianchi_dict[k]:
            w=-wght_of_ind(k.base[k.indices+j],g)
            if not w in syzygies_by_wght: syzygies_by_wght[w]=[]
            new_syzygy=simplify(k.base[k.indices+j]-Bianchi_dict[k][j])
            if new_syzygy!=0:
                new_syzygy=new_syzygy.as_numer_denom()[0]
                syzygies_by_wght[w].append(new_syzygy)

In [ ]:
syzygies_by_wght.keys()

In [ ]:
for a in syzygies_by_wght[9]:
    display(a)

In [26]:
not_added_by_wght={}
for a in not_added:
    w0,w1,w2,w3=[g.basis[i].wght for i in a]
    w=-w0-w1-w2+w3
    if w not in not_added_by_wght: not_added_by_wght[w] = []
    not_added_by_wght[w].append(a)


In [27]:
def add_expr_to_subs_dict(expr,subs_dict):
    expr=expand(ds_subs(expr,Bianchi_dict,D).subs(subs_dict))
    if expr!=0:
        i=0
        new_key=expr.as_coeff_add()[1][i].as_coeff_Mul()[1]
        while type(new_key)==Pow and new_key.as_base_exp()[0]==eta[6,9,9]:
            i+=1
            new_key=expr.as_coeff_add()[1][i].as_coeff_Mul()[1]
        new_val=solve(expr,new_key)[0]
        back_subs({new_key:new_val},subs_dict)
        subs_dict[new_key]=new_val

def back_subs(new_dict,old_dict):
    for k in old_dict:
        old_dict[k]=old_dict[k].subs(new_dict)

In [ ]:
new_subs={}
for w in not_added_by_wght:
    for a in not_added_by_wght[w]:
        expr=simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D))
        add_expr_to_subs_dict(expr,new_subs)

In [ ]:
for a in not_added_by_wght[18]:
    display(simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D)))

### Branching

In [10]:
not_added_by_wght={}
for a in not_added:
    w0,w1,w2,w3=[g.basis[i].wght for i in a]
    w=-w0-w1-w2+w3
    if w not in not_added_by_wght: not_added_by_wght[w] = []
    not_added_by_wght[w].append(a)

In [16]:
not_added_by_wght

{9: [(4, 5, 10, 3)]}

In [11]:
for a in not_added_by_wght[9]:
    display(simplify(ds_subs(P.Bianchi_cache[(a[0],a[1],a[2])].vec[a[3]],Bianchi_dict,D)))

63*(6*eta[6, 9, 9, 3, 4] - eta[6, 9, 9, 4, 3])*eta[6, 9, 9, 4]/275

### First branch

In [35]:
b1_Bianchi_dict=copy.deepcopy(Bianchi_dict)

In [37]:
ds_add_key(eta[6,9,9,4],0,b1_Bianchi_dict,D)

In [ ]:
ds_subs_needed(b1_Bianchi_dict)

In [ ]:
b1_Bianchi_dict[eta[6,9,9]]

### Second branch

In [12]:
b2_Bianchi_dict=copy.deepcopy(Bianchi_dict)

In [13]:
ds_add_key(eta[6,9,9,4,3],6*eta[6,9,9,3,4],b2_Bianchi_dict,D)

In [14]:
ds_subs_needed(Bianchi_dict)

False

In [15]:
b2_Bianchi_dict[eta[6,9,9]]

{(): 0}

In [31]:
# with shelve.open('Abstract_Syzygies') as shelf:
#     shelf['Zero_Wilc_Bianchi_dict'+'{j}'.format(j=7)]=Bianchi_dict

### Ricci Identities

In [131]:
P1=RegularCartanGeometry(g,'eta')
P.curvature=ds_subs(P1.curvature,Bianchi_dict,D)
simplify_cochain(P.curvature)
D.curv=-P.curvature

In [25]:
saved_Bianchi_dict=copy.deepcopy(Bianchi_dict) # Through wght 8+ at the moment
saved_curv=copy.deepcopy(P.curvature)

In [ ]:
duples_by_wght={}
for i in range(3,len(g.basis)):#range(3,len(g.basis)):
    for j in range(i+1,len(g.basis)):
        w=-g.basis[i].wght-g.basis[j].wght
        if w not in duples_by_wght:duples_by_wght[w]=[]
        duples_by_wght[w].append((i,j))

Ricci_dict={}
for w in range(2,7):
    for t in duples_by_wght[w]:
        i,j=t
        time0=time.time()
        temp=ds_subs(dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv),Bianchi_dict,D)
        for k in temp.d:
            temp.d[k]=simplify(temp.d[k])
        temp.clear_zeros()
        if temp.d!={}:
            print('Ricci_Id',(i,j),':',temp,'\n')
            Process_Ricci_Id(D,i,j,Ricci_dict,Bianchi_dict)
            print('\n',(i,j),'complete in time',hrs_min_sec(time.time()-time0),len(list(Ricci_dict.keys())))
            time1=time.time()
            for k in Ricci_dict:
                for j in Ricci_dict[k]:
                    Ricci_dict[k][j]=cancel(Ricci_dict[k][j])
            print('    simplification complete in time',hrs_min_sec(time.time()-time1),'\n')
            print('Ricci_dict:')
            for k in Ricci_dict:
                print('    ',k,'-->',Ricci_dict[k],'\n')
            print('------------------------------------------------------------\n')

In [58]:
def check_viability(t):
    i,j=t
    to_add=dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv)
    to_add=ds_subs(to_add,Bianchi_dict,D)
    viable=False
    for k in to_add.d:
        to_add.d[k]=simplify(to_add.d[k])
        if Indexed_obj_in_expr(to_add.d[k])==set(): viable=True
    return viable

remaining_pairs=set(duples_by_wght[7]+duples_by_wght[8]+duples_by_wght[9])

In [61]:
for t in list(remaining_pairs):
    if check_viability(t):
        i,j=t
        time0=time.time()
        temp=ds_subs(dds_subs(D.Ricci_Id(i,j),Ricci_dict,g,-D.curv),Bianchi_dict,D)
        for k in temp.d:
            temp.d[k]=simplify(temp.d[k])
        temp.clear_zeros()
        if temp.d!={}:
            print('Ricci_Id',(i,j),':',temp,'\n')
            Process_Ricci_Id(D,i,j,Ricci_dict,Bianchi_dict)
            print('\n',(i,j),'complete in time',hrs_min_sec(time.time()-time0),len(list(Ricci_dict.keys())))
            time1=time.time()
            for k in Ricci_dict:
                for j in Ricci_dict[k]:
                    Ricci_dict[k][j]=cancel(Ricci_dict[k][j])
            print('    simplification complete in time',hrs_min_sec(time.time()-time1),'\n')
            print('Ricci_dict:')
            for k in Ricci_dict:
                print('    ',k,'-->',Ricci_dict[k],'\n')
            print('------------------------------------------------------------\n')
        remaining_pairs.remove(t)

In [ ]:
remaining_pairs

In [ ]:
Ricci_dict.keys()

In [ ]:
Bianchi_dict[eta[6,9,9]]

In [ ]:
ds_subs(eta[6,9,9,3,3,4],Bianchi_dict,D)